In [1]:
!python -m spacy download es_core_news_sm

     ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
     ----- ---------------------------------- 1.8/12.9 MB 9.1 MB/s eta 0:00:02
     -------------- ------------------------- 4.7/12.9 MB 11.9 MB/s eta 0:00:01
     --------------------- ------------------ 7.1/12.9 MB 10.9 MB/s eta 0:00:01
     ------------------------------ -------- 10.2/12.9 MB 12.3 MB/s eta 0:00:01
     --------------------------------------  12.8/12.9 MB 12.0 MB/s eta 0:00:01
     --------------------------------------- 12.9/12.9 MB 11.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install --upgrade pip
!pip install numpy pandas
!pip install tf-keras


  Using cached pip-25.3-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-25.3-py3-none-any.whl (1.8 MB)



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Users\alexa\OneDrive\Documentos\GitHub\proyecto_urbania\.venv\Scripts\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
print(os.getcwd())
print(os.listdir())

C:\Users\alexa\OneDrive\Documentos\GitHub\proyecto_urbania
['.git', '.idea', '.venv', 'Capibara_v4.ipynb', 'ciudad_region.csv', 'data', 'df_limpio.csv', 'Extraccion_1.py', 'inmuebles_maria.csv', 'inmuebles_michael.csv', 'inmuebles_prueba.csv', 'inmuebles_prueba_3.csv', 'inmuebles_prueba_4.csv', 'inmuebles_vf.csv', 'logs', 'main.py', 'README.md', 'Recomendacion_urbania.ipynb', 'Recomendacion_urbania.py', 'Recomendacion_urbania_hibrido.ipynb', 'Scrapping']


In [30]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf


BUSQUEDA_CSV = "data/busqueda.csv"
CLIENTES_CSV = "data/clientes.csv"
HISTORIAL_BUSQUEDA = "data/historial_busquedas.csv"

busquedas = pd.read_csv(BUSQUEDA_CSV,
    encoding="latin-1",
    sep=";")
clientes = pd.read_csv(CLIENTES_CSV,    encoding="latin-1",
    sep=";")
historial_busquedas = pd.read_csv(HISTORIAL_BUSQUEDA,    encoding="latin-1",
    sep=";")

busquedas["direccion"] = busquedas["direccion"].fillna("")
busquedas["descripcion"]=busquedas["descripcion"].fillna("")
busquedas["tipo_contrato"] = busquedas["tipo_contrato"].fillna("")
busquedas["tipo_vivienda"] = busquedas["tipo_vivienda"].fillna("")
busquedas["departamento"] = busquedas["departamento"].fillna("")
busquedas["provincia"] = busquedas["provincia"].fillna("")
busquedas["distrito"] = busquedas["distrito"].fillna("")

busquedas["descripcion"] = busquedas["descripcion"] + ' ' + busquedas["tipo_contrato"]+ ' ' + busquedas["tipo_vivienda"]+ ' ' + busquedas["departamento"]+ ' ' + busquedas["provincia"]+ ' ' + busquedas["distrito"]


In [33]:
busquedas.head(1)
busquedas['descripcion'].iloc[0]

'Alquiler Barato Departamento en Los Olivos\nAlquiler de Departamento 3 Dormitorios \x96 Los Olivos | S/ 1,300\nInformacion 9..0.....4......1...6......0.....9...8.......7\nUbicación: Ca. Los Castaños 1317, Los Olivos\nZona residencial tranquila y segura, con excelente conectividad y servicios cerca.\nCaracterísticas del departamento:\n3 dormitorios con buena iluminación natural\nDormitorio principal con baño propio\n3 baños en total\nSala comedor amplia\nCocina con muebles altos y bajos\nLavadero\nConexión a gas natural\nCochera disponible en la calle\n6to piso\nPrecio de alquiler: S/ 1,200\nMantenimiento: S/ 100\nCondiciones: 2 x 1\n** alquiler departamento lima lima los olivos'

In [34]:
print("Detalle por busqueda:", busquedas.shape)
print("Clientes:", clientes.shape)
print("Historial de busquedas:", historial_busquedas.shape)

print("Columnas busquedas:", busquedas.columns.tolist())
print("Columnas clientes:", clientes.columns.tolist())
print("Columnas historial busqueda:", historial_busquedas.columns.tolist())

Detalle por busqueda: (1673, 20)
Clientes: (4, 6)
Historial de busquedas: (216, 4)
Columnas busquedas: ['id_busqueda', 'url', 'distrito_original', 'direccion', 'descripcion', 'fecha_extraccion', 'otros', 'tipo_contrato', 'tipo_vivienda', 'precio', 'area_total', 'area_cubierta', 'banos', 'medios_banos', 'cochera', 'dormitorio', 'antiguedad', 'departamento', 'provincia', 'distrito']
Columnas clientes: ['id_cliente', 'nombre', 'email', 'ciudad', 'pais', 'fecha_registro']
Columnas historial busqueda: ['id_historial', 'id_cliente', 'id_busqueda', 'fecha_compra']


In [35]:
# Merge historial de busqueda con detalle de busqueda
busqueda_detallada = historial_busquedas.merge(busquedas, on="id_busqueda", how="left")
print("Dataset historial por busqueda unido:", busqueda_detallada.shape)
busqueda_detallada.head()

Dataset historial por busqueda unido: (216, 23)


,id_historial,id_cliente,id_busqueda,fecha_compra,url,distrito_original,direccion,descripcion,fecha_extraccion,otros,...,area_total,area_cubierta,banos,medios_banos,cochera,dormitorio,antiguedad,departamento,provincia,distrito
0,1.0,1.0,142.0,22/05/2025,https://urbania.pe/inmueble/clasificado/alclap...,Jesús María,"JR Inca Ripac al 272, Residencial San Felipe,...",Alquiler Minidepartamento Amoblado en Jesus Ma...,"45969,60941","JR Inca Ripac al 272, Residencial San Felipe",...,44.0,44.0,1.0,0.0,1.0,1.0,0.0,lima,lima,jesus maria
1,2.0,1.0,145.0,27/05/2025,https://urbania.pe/inmueble/clasificado/alclap...,Jesús María,"Av. Brasil 2477, Jesús María, Residencial San...",Alquiler de Departamento en Jesús María con Pi...,"45969,60965","Av. Brasil 2477, Jesús María, Residencial San ...",...,57.0,57.0,2.0,0.0,0.0,2.0,7.0,lima,lima,jesus maria
2,3.0,1.0,489.0,28/05/2025,https://urbania.pe/inmueble/clasificado/alclap...,Jesús María,"Prolongacion quito, Oyague, Jesús María",Alquilo Departamento 3Dorm.\nBeatriz .9.9.2.7....,"45969,67583","Prolongacion quito, Oyague",...,90.0,90.0,2.0,0.0,1.0,3.0,5.0,lima,lima,jesus maria
3,4.0,1.0,876.0,29/05/2025,https://urbania.pe/inmueble/clasificado/alclap...,Lima,"AT CDRA 2 DE AV SAN FELIPE, Jesús María, Lima","Jesus Maria 1 Dormitorio 1,200 Soles\nLINDO DE...","45954,0064",AT CDRA 2 DE AV SAN FELIPE,...,135.0,1.0,1.0,0.0,0.0,1.0,1.0,lima,lima,jesus maria
4,5.0,1.0,878.0,4/06/2025,https://urbania.pe/inmueble/clasificado/alclap...,Lima,"Jr lloque yupanqui 1223 , Jesús María, Lima",Mini Departamento Amoblado Wifi Lavadora Terma...,"45954,00676",Jr lloque yupanqui 1223,...,20.0,20.0,1.0,0.0,0.0,1.0,10.0,lima,lima,jesus maria


In [36]:
# Merge con clients
busqueda_full = busqueda_detallada.merge(clientes, on="id_cliente", how="left")
print("Dataset busquedas full:", busqueda_full.shape)
busqueda_full.head()

Dataset busquedas full: (216, 28)


,id_historial,id_cliente,id_busqueda,fecha_compra,url,distrito_original,direccion,descripcion,fecha_extraccion,otros,...,dormitorio,antiguedad,departamento,provincia,distrito,nombre,email,ciudad,pais,fecha_registro
0,1.0,1.0,142.0,22/05/2025,https://urbania.pe/inmueble/clasificado/alclap...,Jesús María,"JR Inca Ripac al 272, Residencial San Felipe,...",Alquiler Minidepartamento Amoblado en Jesus Ma...,"45969,60941","JR Inca Ripac al 272, Residencial San Felipe",...,1.0,0.0,lima,lima,jesus maria,Mateo Rojas,mateo.r@email.com,Cusco,Peru,25/10/2025
1,2.0,1.0,145.0,27/05/2025,https://urbania.pe/inmueble/clasificado/alclap...,Jesús María,"Av. Brasil 2477, Jesús María, Residencial San...",Alquiler de Departamento en Jesús María con Pi...,"45969,60965","Av. Brasil 2477, Jesús María, Residencial San ...",...,2.0,7.0,lima,lima,jesus maria,Mateo Rojas,mateo.r@email.com,Cusco,Peru,25/10/2025
2,3.0,1.0,489.0,28/05/2025,https://urbania.pe/inmueble/clasificado/alclap...,Jesús María,"Prolongacion quito, Oyague, Jesús María",Alquilo Departamento 3Dorm.\nBeatriz .9.9.2.7....,"45969,67583","Prolongacion quito, Oyague",...,3.0,5.0,lima,lima,jesus maria,Mateo Rojas,mateo.r@email.com,Cusco,Peru,25/10/2025
3,4.0,1.0,876.0,29/05/2025,https://urbania.pe/inmueble/clasificado/alclap...,Lima,"AT CDRA 2 DE AV SAN FELIPE, Jesús María, Lima","Jesus Maria 1 Dormitorio 1,200 Soles\nLINDO DE...","45954,0064",AT CDRA 2 DE AV SAN FELIPE,...,1.0,1.0,lima,lima,jesus maria,Mateo Rojas,mateo.r@email.com,Cusco,Peru,25/10/2025
4,5.0,1.0,878.0,4/06/2025,https://urbania.pe/inmueble/clasificado/alclap...,Lima,"Jr lloque yupanqui 1223 , Jesús María, Lima",Mini Departamento Amoblado Wifi Lavadora Terma...,"45954,00676",Jr lloque yupanqui 1223,...,1.0,10.0,lima,lima,jesus maria,Mateo Rojas,mateo.r@email.com,Cusco,Peru,25/10/2025


In [57]:
numeric_cols = ["precio","area_total","banos","cochera","dormitorio","antiguedad"]

In [58]:
client_profiles = (
    busqueda_full.groupby("id_cliente")
    .apply(lambda g: " ".join(g["direccion"].tolist() + g["descripcion"].tolist()))
    .reset_index(name="texto_cliente")
)

# Agregar información de clientes (ciudad, país)
client_profiles = client_profiles.merge(clientes, on="id_cliente")
client_profiles["texto_cliente"] = (
    client_profiles["nombre"] + " " +
    client_profiles["ciudad"] + " " +
    client_profiles["pais"] + " " +
    client_profiles["texto_cliente"]
)

print("Perfiles de clientes generados:", client_profiles.shape)
client_profiles.head()

Perfiles de clientes generados: (3, 7)


C:\Users\alexa\AppData\Local\Temp\ipykernel_21512\2464208606.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: " ".join(g["direccion"].tolist() + g["descripcion"].tolist()))


,id_cliente,texto_cliente,nombre,email,ciudad,pais,fecha_registro
0,1.0,"Mateo Rojas Cusco Peru JR Inca Ripac al 272, ...",Mateo Rojas,mateo.r@email.com,Cusco,Peru,25/10/2025
1,2.0,Luis HernÃ¡ndez Lima Peru Calle 30 #300 San Bo...,Luis HernÃ¡ndez,luis.h@email.com,Lima,Peru,15/10/2025
2,3.0,Yury Aquino Lima Peru CALLE PARQUE LAS ORQUIDE...,Yury Aquino,yury300365@hotmail.com,Lima,Peru,20/10/2025


In [59]:
from sklearn.preprocessing import StandardScaler

# Asegurarnos de que las columnas numéricas existan y no tengan NaN
for col in numeric_cols:
    if col not in busquedas.columns:
        raise ValueError(f"La columna numérica '{col}' no existe en busquedas")
    busquedas[col] = pd.to_numeric(busquedas[col], errors="coerce").fillna(0)

# 1) Matriz numérica de BÚSQUEDAS (una fila por busqueda)
busq_num = busquedas[numeric_cols].values  # shape: (n_busquedas, n_numeric)

# 2) Perfil numérico de cada CLIENTE: promedio de lo que ya buscó
#    Usamos busqueda_full (historial + detalle + cliente)
client_num_df = (
    busqueda_full
    .groupby("id_cliente")[numeric_cols]
    .mean()
    .reindex(client_profiles["id_cliente"])   # mismo orden que client_profiles
    .fillna(0)
)

client_num = client_num_df.values  # shape: (n_clientes, n_numeric)

# 3) Escalar todo junto (mismo scaler para clientes y búsquedas)
scaler = StandardScaler()
busq_num_scaled = scaler.fit_transform(busq_num)
client_num_scaled = scaler.transform(client_num)


In [60]:
# Remover Stopwords y Lematizar en español en texto_cliente
import spacy
nlp = spacy.load("es_core_news_sm")
print("spaCy cargado: usando lematización y stopwords de spaCy.")

def preprocess_spacy(text):
    doc = nlp(str(text).lower())
    tokens = [t.lemma_ for t in doc if not t.is_stop and t.is_alpha]
    return " ".join(tokens)

# Procesar texto de clientes
client_profiles["texto_cliente_proc"] = client_profiles["texto_cliente"].apply(preprocess_spacy)

# Revisa resultados
client_profiles[["id_cliente", "nombre", "texto_cliente", "texto_cliente_proc"]].head()

spaCy cargado: usando lematización y stopwords de spaCy.


,id_cliente,nombre,texto_cliente,texto_cliente_proc
0,1.0,Mateo Rojas,"Mateo Rojas Cusco Peru JR Inca Ripac al 272, ...",mateo rojo cusco peru jr inco ripac residencia...
1,2.0,Luis HernÃ¡ndez,Luis HernÃ¡ndez Lima Peru Calle 30 #300 San Bo...,luis lima peru calle san borja san borja limo ...
2,3.0,Yury Aquino,Yury Aquino Lima Peru CALLE PARQUE LAS ORQUIDE...,yury aquino lima peru calle parcar orquidea pu...


In [61]:
from sentence_transformers import SentenceTransformer

print("SentenceTransformers disponible. Usando modelo multilingüe.")
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
st_model = SentenceTransformer(model_name)

SentenceTransformers disponible. Usando modelo multilingüe.


In [62]:
import random


# Preprocesar busquedas con spaCy
busquedas["busqueda_direccion"] = busquedas["direccion"].apply(preprocess_spacy)
busquedas["busqueda_descripcion"] = busquedas["descripcion"].apply(preprocess_spacy)

# Listas de textos
direccion_textos = busquedas["busqueda_direccion"].tolist()
descripcion_textos = busquedas["busqueda_descripcion"].tolist()
clientes_textos = client_profiles["texto_cliente_proc"].tolist()

# Todos los textos para embeddings
todos_textos = direccion_textos + descripcion_textos + clientes_textos

# Mostrar 5 elementos aleatorios
muestra_aleatoria = random.sample(todos_textos, 5)

print("Muestra aleatoria de la lista 'todos_textos':")
for i, texto in enumerate(muestra_aleatoria):
    print(f"{i+1}. {texto}")

Muestra aleatoria de la lista 'todos_textos':
1. victor maurtua san isidro lima
2. alquiler almacir local industrial sachaco arequipar ubicación carretera variante uchumayo km sachaco arequipar frente grifo pecsa cural m cruce vía evitamiento via acceso variante uchumayo conectar distrito cerro colorado sachacha uchumayo cercado arequipa yanahuar joya parque industrial arequipa centro minero cerro verde vía evitamiento sierra sur bolivia panamericana sur lindero medida perimétrica frente ml carretera variante uchumayo km derecha ml propiedad fl smith izquierda ml propiedad tercero fondo ml propiedad tercero local industrial línea producción línea producción armado soldadura línea producción liberación granallado pintura línea habilitar roladoro área almacenaje accesorio almacén suministro materia primo techado terreno tijeral metálico altura aproximado nave cerramiento vertical perímetro oficina portátil contenedor comedor obrero pozo agua bomba sumergible corriente eléctrico capacidad

In [63]:
# Construir textos para ambos grupos
clientes_textos = client_profiles["texto_cliente"].tolist()
busqueda_textos = busquedas["direccion"].tolist()

# Concatenar para un solo encode
todos_textos = clientes_textos + busqueda_textos

# Generar embeddings de todo junto
embeddings = st_model.encode(todos_textos, convert_to_numpy=True, show_progress_bar=True)

# Separar los embeddings en clientes y productos
n_clientes = len(clientes_textos)
n_busquedas = len(busqueda_textos)

print(len(embeddings))
print((n_clientes))
print((n_busquedas))

client_embeddings = embeddings[:n_clientes]
busqueda_embeddings = embeddings[n_clientes:]

Batches: 100%|██████████| 53/53 [00:21<00:00,  2.45it/s]

1676
3
1673


In [64]:
out_dir = "logs/embeddings"
os.makedirs(out_dir, exist_ok=True)

def make_label(text, n_words=5):
    return " ".join(text.split()[:n_words])

# Concatenar embeddings (clientes + productos)
all_embeddings = np.vstack([client_embeddings, busqueda_embeddings])

# Crear metadata
metadata = pd.DataFrame({
    "label": (
        [make_label(t, 5) for t in client_profiles["texto_cliente"].tolist()] +
        [make_label(t, 5) for t in busquedas["direccion"].tolist()]
    ),
    "tipo": (
        ["cliente"] * len(client_profiles) +
        ["direccion"] * len(busquedas)
    ),
    "texto_completo": (
        client_profiles["texto_cliente"].tolist() +
        busquedas["direccion"].tolist()
    )
})

# Guardar metadata como TSV
metadata_path = os.path.join(out_dir, "metadata.tsv")
metadata.to_csv(metadata_path, sep="\t", index=False)

print("Metadata guardada en:", metadata_path)

# Guardar embeddings para TensorBoard
emb_var = tf.Variable(all_embeddings, name="embeddings")
ckpt = tf.train.Checkpoint(embedding=emb_var)
ckpt.save(os.path.join(out_dir, "embedding.ckpt"))

print("Embeddings guardados en:", out_dir)
print("Ejecuta en terminal: tensorboard --logdir logs/embeddings")

Metadata guardada en: logs/embeddings\metadata.tsv
Embeddings guardados en: logs/embeddings
Ejecuta en terminal: tensorboard --logdir logs/embeddings


In [82]:
# Normalizar embeddings de clientes y productos (una vez)
client_embeddings_norm = client_embeddings / np.linalg.norm(client_embeddings, axis=1, keepdims=True)
busqueda_embeddings_norm = busqueda_embeddings / np.linalg.norm(busqueda_embeddings, axis=1, keepdims=True)

# Pesos para cada parte
alpha_text = 0.8  # peso de la parte de texto
beta_num   = 0.2  # peso de la parte numérica

# Embeddings híbridos de BÚSQUEDAS
busqueda_embeddings_hybrid = np.hstack([
    alpha_text * busqueda_embeddings_norm,
    beta_num   * busq_num_scaled
])

# Embeddings híbridos de CLIENTES
client_embeddings_hybrid = np.hstack([
    alpha_text * client_embeddings_norm,
    beta_num   * client_num_scaled
])


In [83]:
def top_k_products_for_client_hybrid(client_idx, k=3):
    """
    Dado un índice de cliente, devuelve los k sitios más similares
    usando TEXTO + VARIABLES NUMÉRICAS, excluyendo lo que ya buscó.
    """
    # Vector híbrido del cliente
    q = client_embeddings_hybrid[client_idx]  # (dim_total,)

    # Similaridad por producto (búsqueda)
    sims = np.dot(busqueda_embeddings_hybrid, q)  # (n_busquedas,)

    # Busquedas ya realizadas por este cliente
    cliente_id = client_profiles.iloc[client_idx]["id_cliente"]
    buscados = set(
        busqueda_full.loc[busqueda_full["id_cliente"] == cliente_id, "id_busqueda"].tolist()
    )

    # Ordenar por similitud (descendente)
    topk_idx = sims.argsort()[::-1]

    # Filtrar búsquedas que ya vio
    recomendados = []
    for idx in topk_idx:
        pid = busquedas.iloc[idx]["id_busqueda"]
        if pid not in buscados:
            recomendados.append((
                idx,
                sims[idx],
                pid,
                busquedas.iloc[idx]["direccion"],
                busquedas.iloc[idx]["descripcion"],
                *[busquedas.iloc[idx][col] for col in numeric_cols]  # añadir numéricas
            ))
        if len(recomendados) >= k:
            break

    return recomendados


In [84]:
cliente_idx = 1
cliente_id = client_profiles.iloc[cliente_idx]["id_cliente"]
print(f"Cliente {cliente_id}: {client_profiles.iloc[cliente_idx]['texto_cliente'][:120]}...")
print("Top-3 búsquedas recomendadas (híbrido TEXTO + NUMÉRICOS, no visitadas):")

for rec in top_k_products_for_client_hybrid(cliente_idx, k=3):
    idx, score, pid, direccion, desc, *nums = rec
    num_info = ", ".join(f"{col}={val}" for col, val in zip(numeric_cols, nums))
    print(f" - id_busqueda={pid}, score={score:.4f}")
    print(f"   dirección: {direccion}")
    print(f"   desc: {desc[:80]}...")
    print(f"   {num_info}")
    print()

Cliente 2.0: Luis HernÃ¡ndez Lima Peru Calle 30 #300 San Borja,  San Borja, Lima Rousseau 264,  San Borja, Lima San Borja,  San Borja...
Top-3 búsquedas recomendadas (híbrido TEXTO + NUMÉRICOS, no visitadas):
 - id_busqueda=571, score=0.5464
   dirección: CALLE DURERO 257, SAN BORJA,  San Borja, Lima
   desc: Se Alquila Linda Casa 4 Dorm. - Calle Durero, San Borja S/.7,300
ALQUILO CASA EN...
   precio=7300.0, area_total=336.0, banos=3, cochera=2, dormitorio=4, antiguedad=40

 - id_busqueda=191, score=0.5401
   dirección: SAN Borja Norte 459,  San Borja, Lima
   desc: Tranquilidad y Conexión en San Borja Norte: Alquiler Departamento Amoblado, Gas ...
   precio=3053.0, area_total=93.0, banos=2, cochera=1, dormitorio=3, antiguedad=10

 - id_busqueda=572, score=0.5392
   dirección: Av. San Borja Norte 800,  San Borja, Lima
   desc: Se Alquila Casa en Av - Cz - 2 Pisos en San Borja Norte
Casa Comercial en Av. Sa...
   precio=16428.0, area_total=352.0, banos=3, cochera=5, dormitorio=6, antig